## Filter-based feature selection (no model training)

Each cell ranks columns by mutual information with cancellation using the training period only. Mutual information is a filter score: no classifier is fitted. Each block keeps the top half of its candidate columns as an exploratory subset. The all-source block excludes `is_canceled` from inputs; it includes outcome-related and post-booking columns only as a leakage diagnostic, never as a deployable feature list. Validate a final set on choose data, and treat the already-inspected proof period as a secondary evaluation.


In [4]:
from pathlib import Path
import sys
import math

import pandas as pd
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif

# Support running this notebook with the project root or notebooks/ as the working directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import contract
import data
import features

raw_df = data.read_raw()
prepared_df = features.prepare_bookings(raw_df)
train_df, choose_df, proof_df = contract.split_periods(prepared_df)
y_train = train_df[features.TARGET]

def rank_features_by_mutual_information(candidate_columns, set_name):
    """Rank original columns using training-period mutual information only."""
    rows = []
    for column in candidate_columns:
        values = train_df[column]
        if pd.api.types.is_numeric_dtype(values):
            numeric_values = values.fillna(values.median()).to_numpy(dtype=float).reshape(-1, 1)
            score = mutual_info_classif(
                numeric_values, y_train, discrete_features=False, random_state=42
            )[0]
        else:
            category_codes = pd.factorize(
                values.fillna("unknown").astype(str), sort=True
            )[0].reshape(-1, 1)
            score = mutual_info_classif(
                category_codes, y_train, discrete_features=True, random_state=42
            )[0]
        rows.append({"Feature": column, "Mutual information": float(score)})

    ranking = pd.DataFrame(rows).sort_values(
        "Mutual information", ascending=False
    ).reset_index(drop=True)
    keep_count = max(1, math.ceil(len(candidate_columns) / 2))
    ranking.insert(0, "Feature set", set_name)
    ranking["Selected"] = ranking.index < keep_count
    return ranking

# Block 1: all source columns (32 total, minus is_canceled as the target = 31 predictors).
all_source_predictors = [
    column for column in raw_df.columns
    if column != features.TARGET
]
all_columns_selection_df = rank_features_by_mutual_information(
    all_source_predictors, "All 31 source predictors"
)
all_columns_selected = all_columns_selection_df.loc[
    all_columns_selection_df["Selected"], "Feature"
].tolist()
print("Source columns:", len(raw_df.columns), "including the target")
print("Predictors ranked:", len(all_source_predictors))
print("Selected features:", all_columns_selected)
display(all_columns_selection_df)


Source columns: 32 including the target
Predictors ranked: 31
Selected features: ['reservation_status', 'reservation_status_date', 'deposit_type', 'lead_time', 'previous_cancellations', 'adr', 'country', 'agent', 'market_segment', 'assigned_room_type', 'total_of_special_requests', 'days_in_waiting_list', 'required_car_parking_spaces', 'booking_changes', 'distribution_channel', 'hotel']


,Feature set,Feature,Mutual information,Selected
0,All 31 source predictors,reservation_status,0.653596,True
1,All 31 source predictors,reservation_status_date,0.202442,True
2,All 31 source predictors,deposit_type,0.173386,True
3,All 31 source predictors,lead_time,0.128023,True
4,All 31 source predictors,previous_cancellations,0.118255,True
5,All 31 source predictors,adr,0.113767,True
6,All 31 source predictors,country,0.113420,True
7,All 31 source predictors,agent,0.102697,True
8,All 31 source predictors,market_segment,0.050290,True
9,All 31 source predictors,assigned_room_type,0.046958,True


In [5]:
# Block 2: the team's approved 17 model features.
seventeen_feature_selection_df = rank_features_by_mutual_information(
    features.MODEL_FEATURES, "Approved 17 features"
)
seventeen_features_selected = seventeen_feature_selection_df.loc[
    seventeen_feature_selection_df["Selected"], "Feature"
].tolist()
print("Approved features ranked:", len(features.MODEL_FEATURES))
print("Selected features:", seventeen_features_selected)
display(seventeen_feature_selection_df)


Approved features ranked: 17
Selected features: ['lead_time', 'previous_cancellations', 'country', 'agent', 'market_segment', 'distribution_channel', 'hotel', 'arrival_date_week_number', 'company']


,Feature set,Feature,Mutual information,Selected
0,Approved 17 features,lead_time,0.128023,True
1,Approved 17 features,previous_cancellations,0.118255,True
2,Approved 17 features,country,0.113420,True
3,Approved 17 features,agent,0.102697,True
4,Approved 17 features,market_segment,0.050290,True
5,Approved 17 features,distribution_channel,0.015464,True
6,Approved 17 features,hotel,0.013781,True
7,Approved 17 features,arrival_date_week_number,0.013436,True
8,Approved 17 features,company,0.012563,True
9,Approved 17 features,reserved_room_type,0.008193,False


In [6]:
# Block 3: the nine fields classified as Bin 1 in docs/COLUMNS.md.
bin1_features = [
    "hotel",
    "market_segment",
    "distribution_channel",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "agent",
    "company",
    "customer_type",
]
nine_feature_selection_df = rank_features_by_mutual_information(
    bin1_features, "Nine Bin 1 candidates"
)
nine_features_selected = nine_feature_selection_df.loc[
    nine_feature_selection_df["Selected"], "Feature"
].tolist()
print("Bin 1 candidates ranked:", len(bin1_features))
print("Selected features:", nine_features_selected)
display(nine_feature_selection_df)


Bin 1 candidates ranked: 9
Selected features: ['previous_cancellations', 'agent', 'market_segment', 'distribution_channel', 'hotel']


,Feature set,Feature,Mutual information,Selected
0,Nine Bin 1 candidates,previous_cancellations,0.118255,True
1,Nine Bin 1 candidates,agent,0.102697,True
2,Nine Bin 1 candidates,market_segment,0.050290,True
3,Nine Bin 1 candidates,distribution_channel,0.015464,True
4,Nine Bin 1 candidates,hotel,0.013781,True
5,Nine Bin 1 candidates,company,0.012563,False
6,Nine Bin 1 candidates,previous_bookings_not_canceled,0.004410,False
7,Nine Bin 1 candidates,customer_type,0.003717,False
8,Nine Bin 1 candidates,is_repeated_guest,0.000000,False
